# ECM generation

`generate_ecm` runs a virtual pulse characterisation on the physics model and fits an n-RC network to it, so the equivalent circuit model inherits the physics of the exact design you are working on.

## 1. A fresh ECM for a design

`ecm_options` sets up the fit, temperatures are in Celsius and any grid you leave out uses the backend default.

In [ ]:
from breathe_simulate import api_interface as api
from breathe_simulate.ecm import ecm_options

cell_name = "Molicel P45B"

options = ecm_options(
    n_rc=2,
    soc_grid=[0.1, 0.5, 0.9],
    temp_grid_degC=[10.0, 25.0],
    charge_c_rates=[1.0],
    discharge_c_rates=[1.0, 3.0],
)

fresh = api.generate_ecm(cell_name, ecm=options)
fresh

### The parameter tables

`to_dataframe()` flattens the fitted tables into one long-format hand-off table, and a `run_sim` style dict passed as `design=...` characterises a modified cell instead.

In [ ]:
table = fresh.to_dataframe()
table.head(12)

`r0`, `r(k)` and `c(k)` return the raw 3-D arrays over (SoC x temperature x C-rate).

In [ ]:
print("R0 discharge table shape:", fresh.r0("discharge").shape)
print("SoC grid:", fresh.soc_grid)
print("Temperature grid [degC]:", fresh.temp_grid_degC)
print("Capacity [Ah]:", fresh.capacity)

fresh.plot_parameters("R0", direction="discharge", c_rate=1.0)

### OCV, thermal constants and saving

The result also carries the OCV curves and thermal constants an ECM runtime needs, so it is a complete self-contained model.

In [ ]:
fresh.ocv.head()

In [ ]:
fresh.thermal

In [ ]:
from breathe_simulate.results import EcmResults

fresh.save("fresh_ecm.json")
reloaded = EcmResults.load("fresh_ecm.json")
reloaded

## 2. An aged ECM from a degradation campaign

Pass `ecm=True` (or an `ecm_options(...)` block) to `run_ageing_sim` and the same fit runs at the end-of-campaign aged state, with the campaign's final degradation applied to the physics.

In [ ]:
from breathe_simulate.ageing import AgeingCycler, RptCycler

CAP_AH = 4.5

ageing = AgeingCycler(selected_unit="C", cell_capacity=CAP_AH).cyclic(
    I_chg=1.0,
    I_dch=-1.0,
    I_cut=0.05,
    V_max=4.2,
    V_min=2.5,
    t_rest_s=300,
    t_max_cv_s=3600,
)
rpt = RptCycler(selected_unit="C", cell_capacity=CAP_AH).build(
    I_chg=1.0,
    I_cut=0.05,
    V_max=4.2,
    V_min=2.5,
    capacity_checks=[{"current": 1.0, "reference": True}],
    pulses=[{"soc": 0.5, "current": -1.0, "duration_s": 30, "reference": True}],
    t_equilibration_s=1800,
)

result = api.run_ageing_sim(
    cell_name,
    ageing,
    rpt_cycler=rpt,
    max_cycles=200,
    rpt_every_n_cycles=50,
    ecm=options,
)

aged = result.ecm
aged

`degradation_state` records the aged state the fit ran at.

In [ ]:
aged.degradation_state

## 3. Fresh against aged

Both fits used the same grids, so the tables compare point for point.

In [ ]:
import plotly.graph_objects as go

temp_index = 1  # 25 degC in the grid above
rate_index = 0  # 1C discharge pulses

fig = go.Figure()
for label, ecm in (("fresh", fresh), ("aged", aged)):
    fig.add_trace(
        go.Scatter(
            x=ecm.soc_grid,
            y=1000 * ecm.r0("discharge")[:, temp_index, rate_index],
            mode="lines+markers",
            name=label,
        )
    )
fig.update_layout(
    title="R0 vs SoC at 25 degC, 1C discharge pulses",
    xaxis_title="SoC",
    yaxis_title="R0 [mOhm]",
    legend_title="Cell state",
)
fig

The long-format tables take the comparison further, for example the relative R0 growth at every operating point.

In [ ]:
fresh_table = fresh.to_dataframe().set_index(
    ["soc", "temperature_degC", "c_rate", "direction"]
)
aged_table = aged.to_dataframe().set_index(
    ["soc", "temperature_degC", "c_rate", "direction"]
)

growth = (aged_table["R0_Ohm"] / fresh_table["R0_Ohm"] - 1) * 100
growth.rename("R0 growth [%]").reset_index().head(12)

## Where this fits

Export `to_dataframe()` to CSV for your ECM toolchain, or `save()` the full JSON and reload it later with `EcmResults.load`.